# 🎬 Netflix Top Shows & Movies — Beginner Data Science Project

**Welcome!** This notebook is a beginner-friendly, step-by-step guide to exploring a Netflix dataset.

By the end of this project you will be able to:
- Load and inspect a real-world dataset
- Clean messy data
- Answer interesting questions with code
- Create clear charts and visualizations

---

## Table of Contents
1. [Setup — Import Libraries](#1)
2. [Load the Dataset](#2)
3. [Explore the Data](#3)
4. [Data Cleaning](#4)
5. [Exploratory Data Analysis (EDA)](#5)
6. [Visualizations](#6)
7. [Insights & Conclusion](#7)
8. [Your Turn — Practice Exercises](#8)

---
## 1. Setup — Import Libraries <a id="1"></a>

We use four popular Python libraries:

| Library | Purpose |
|---------|----------|
| `pandas` | Load, clean and analyse tabular data |
| `numpy` | Fast numerical operations |
| `matplotlib` | Create basic charts |
| `seaborn` | Create beautiful statistical charts |

In [ ]:
# Import the libraries we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Make charts appear inline in the notebook
%matplotlib inline

# Set a visual style for all charts
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

print("✅ Libraries imported successfully!")

---
## 2. Load the Dataset <a id="2"></a>

Our dataset is stored in a CSV file called `netflix_top_shows_2023.csv`.  
CSV stands for **Comma-Separated Values** — it is just a plain text file where each row is a record.

In [ ]:
# Load the CSV file into a pandas DataFrame
df = pd.read_csv("netflix_top_shows_2023.csv")

# How many rows and columns do we have?
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Look at the first 5 rows
df.head()

In [ ]:
# Look at the last 5 rows
df.tail()

### Column Descriptions

| Column | Description |
|--------|-------------|
| `show_id` | Unique identifier |
| `title` | Name of the show or movie |
| `type` | TV Show or Movie |
| `director` | Director(s) |
| `cast` | Main actors |
| `country` | Country of production |
| `date_added` | When it was added to Netflix |
| `release_year` | Original release year |
| `rating` | Age/content rating (e.g. TV-MA, PG-13) |
| `duration` | Length (minutes for movies, seasons for TV) |
| `genre` | Genre(s) |
| `imdb_score` | IMDb user rating (1–10) |
| `votes` | Number of IMDb votes |
| `description` | Short plot summary |

---
## 3. Explore the Data <a id="3"></a>

Before cleaning anything, always **explore first** to understand what you are working with.

In [ ]:
# Get the data types of each column
df.dtypes

In [ ]:
# Count missing (NaN) values in each column
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Summary statistics for numeric columns
df.describe()

In [ ]:
# How many TV Shows vs Movies?
print("Content type breakdown:")
print(df["type"].value_counts())

In [ ]:
# What unique age ratings exist?
print("Unique age ratings:")
print(sorted(df["rating"].dropna().unique()))

In [ ]:
# Which countries appear most often?
print("Top 10 countries by number of titles:")
print(df["country"].value_counts().head(10))

---
## 4. Data Cleaning <a id="4"></a>

Real-world data is never perfect. We need to:
- Handle missing values
- Fix data types
- Extract useful information from text columns

In [ ]:
# Make a copy so the original data is not changed
df_clean = df.copy()

# Fill missing director and cast with 'Unknown'
df_clean["director"] = df_clean["director"].fillna("Unknown")
df_clean["cast"] = df_clean["cast"].fillna("Unknown")

# Drop rows where imdb_score is missing (we need it for analysis)
before = len(df_clean)
df_clean = df_clean.dropna(subset=["imdb_score"])
after = len(df_clean)
print(f"Removed {before - after} rows with missing IMDb score. {after} rows remain.")

In [ ]:
# Convert date_added to a proper datetime type
df_clean["date_added"] = pd.to_datetime(df_clean["date_added"], format="%B %d %Y", errors="coerce")

# Extract the year and month Netflix added the title
df_clean["year_added"] = df_clean["date_added"].dt.year
df_clean["month_added"] = df_clean["date_added"].dt.month

print("Date columns created:")
df_clean[["title", "date_added", "year_added", "month_added"]].head()

In [ ]:
# For Movies, extract just the number of minutes from the 'duration' column
# Example: '117 min' → 117
movies = df_clean[df_clean["type"] == "Movie"].copy()
movies["duration_min"] = movies["duration"].str.extract(r"(\d+)").astype(float)

print("Movie duration stats (minutes):")
print(movies["duration_min"].describe())

In [ ]:
# For TV Shows, extract the number of seasons
# Example: '3 Seasons' → 3, '1 Season' → 1
shows = df_clean[df_clean["type"] == "TV Show"].copy()
shows["num_seasons"] = shows["duration"].str.extract(r"(\d+)").astype(float)

print("TV Show seasons stats:")
print(shows["num_seasons"].describe())

In [ ]:
# Check data after cleaning
print(f"Clean dataset: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
df_clean.head(3)

---
## 5. Exploratory Data Analysis (EDA) <a id="5"></a>

Now let's answer specific questions using our clean data.

### 5.1 What are the highest-rated titles on Netflix?

In [ ]:
# Sort by IMDb score (descending)
top10 = df_clean.sort_values("imdb_score", ascending=False).head(10)
top10[["title", "type", "genre", "imdb_score", "votes"]].reset_index(drop=True)

### 5.2 What is the average IMDb score for TV Shows vs Movies?

In [ ]:
avg_score_by_type = df_clean.groupby("type")["imdb_score"].mean().round(2)
print("Average IMDb Score by Content Type:")
print(avg_score_by_type)

### 5.3 Which countries produce the most Netflix content?

In [ ]:
top_countries = df_clean["country"].value_counts().head(10)
print("Top 10 Countries by Number of Titles:")
print(top_countries)

### 5.4 How have Netflix additions changed over the years?

In [ ]:
titles_per_year = df_clean.groupby("year_added").size()
print("Titles added to Netflix per year:")
print(titles_per_year)

### 5.5 What are the most common age ratings?

In [ ]:
rating_counts = df_clean["rating"].value_counts()
print("Age Rating Distribution:")
print(rating_counts)

### 5.6 Which genre has the highest average IMDb score?

In [ ]:
# The 'genre' column can contain multiple genres (e.g., 'Crime Drama')
# We'll use the full genre string as a category
genre_scores = (
    df_clean.groupby("genre")["imdb_score"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "avg_score", "count": "num_titles"})
    .query("num_titles >= 2")  # Only genres with 2+ titles
    .sort_values("avg_score", ascending=False)
    .round(2)
)
print("Top 10 Genres by Average IMDb Score (min. 2 titles):")
print(genre_scores.head(10))

### 5.7 Do movies with more votes tend to have higher scores? (Correlation)

In [ ]:
# Correlation between votes and IMDb score
correlation = df_clean["votes"].corr(df_clean["imdb_score"])
print(f"Pearson correlation between votes and IMDb score: {correlation:.3f}")
print("\nInterpretation:")
if correlation > 0.3:
    print("  → Moderate to strong positive correlation: higher-rated titles tend to attract more voters.")
else:
    print("  → Weak correlation: vote count and score are largely independent.")

---
## 6. Visualizations <a id="6"></a>

Charts make it much easier to communicate findings. Let's create several.

### 6.1 TV Shows vs Movies — Pie Chart

In [ ]:
type_counts = df_clean["type"].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    type_counts,
    labels=type_counts.index,
    autopct="%1.1f%%",
    colors=["#E50914", "#221F1F"],
    startangle=140,
    textprops={"fontsize": 13},
)
ax.set_title("Netflix Content: TV Shows vs Movies", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

### 6.2 Distribution of IMDb Scores — Histogram

In [ ]:
fig, ax = plt.subplots()
ax.hist(df_clean["imdb_score"], bins=15, color="#E50914", edgecolor="white", alpha=0.85)
ax.axvline(df_clean["imdb_score"].mean(), color="black", linestyle="--", linewidth=1.5,
           label=f'Mean = {df_clean["imdb_score"].mean():.2f}')
ax.set_xlabel("IMDb Score", fontsize=12)
ax.set_ylabel("Number of Titles", fontsize=12)
ax.set_title("Distribution of IMDb Scores on Netflix", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

### 6.3 Top 15 Highest-Rated Titles — Horizontal Bar Chart

In [ ]:
top15 = df_clean.nlargest(15, "imdb_score")[["title", "type", "imdb_score"]].sort_values("imdb_score")

colors = ["#E50914" if t == "TV Show" else "#564d4d" for t in top15["type"]]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top15["title"], top15["imdb_score"], color=colors, edgecolor="white")
ax.bar_label(bars, fmt="%.1f", padding=4, fontsize=10)
ax.set_xlim(7, 10)
ax.set_xlabel("IMDb Score", fontsize=12)
ax.set_title("Top 15 Highest-Rated Netflix Titles", fontsize=14, fontweight="bold")

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#E50914", label="TV Show"),
                   Patch(facecolor="#564d4d", label="Movie")]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()

### 6.4 Top 10 Countries by Number of Titles — Bar Chart

In [ ]:
top_countries = df_clean["country"].value_counts().head(10)

fig, ax = plt.subplots()
sns.barplot(x=top_countries.values, y=top_countries.index, palette="Reds_r", ax=ax)
ax.set_xlabel("Number of Titles", fontsize=12)
ax.set_ylabel("Country", fontsize=12)
ax.set_title("Top 10 Countries by Netflix Content Count", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 6.5 IMDb Score Comparison — TV Shows vs Movies (Box Plot)

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(
    data=df_clean,
    x="type",
    y="imdb_score",
    palette={"TV Show": "#E50914", "Movie": "#564d4d"},
    width=0.4,
    ax=ax,
)
ax.set_xlabel("Content Type", fontsize=12)
ax.set_ylabel("IMDb Score", fontsize=12)
ax.set_title("IMDb Score Distribution: TV Shows vs Movies", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 6.6 Age Rating Distribution — Bar Chart

In [ ]:
rating_counts = df_clean["rating"].value_counts()

fig, ax = plt.subplots()
sns.barplot(x=rating_counts.index, y=rating_counts.values, palette="Reds_r", ax=ax)
ax.set_xlabel("Age Rating", fontsize=12)
ax.set_ylabel("Number of Titles", fontsize=12)
ax.set_title("Netflix Content by Age Rating", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 6.7 Movie Duration Distribution — Histogram

In [ ]:
fig, ax = plt.subplots()
ax.hist(movies["duration_min"].dropna(), bins=15, color="#564d4d", edgecolor="white", alpha=0.85)
ax.axvline(movies["duration_min"].mean(), color="#E50914", linestyle="--", linewidth=2,
           label=f'Mean = {movies["duration_min"].mean():.0f} min')
ax.set_xlabel("Duration (minutes)", fontsize=12)
ax.set_ylabel("Number of Movies", fontsize=12)
ax.set_title("Distribution of Netflix Movie Durations", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

### 6.8 Votes vs IMDb Score — Scatter Plot

In [ ]:
fig, ax = plt.subplots()
scatter_colors = df_clean["type"].map({"TV Show": "#E50914", "Movie": "#564d4d"})
ax.scatter(df_clean["votes"] / 1000, df_clean["imdb_score"],
           c=scatter_colors, alpha=0.6, edgecolors="white", linewidth=0.3, s=60)
ax.set_xlabel("IMDb Votes (thousands)", fontsize=12)
ax.set_ylabel("IMDb Score", fontsize=12)
ax.set_title("IMDb Votes vs Score", fontsize=14, fontweight="bold")

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#E50914", label="TV Show"),
                   Patch(facecolor="#564d4d", label="Movie")]
ax.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

### 6.9 Top 10 Genres by Average IMDb Score — Bar Chart

In [ ]:
top_genres = (
    df_clean.groupby("genre")["imdb_score"]
    .agg(["mean", "count"])
    .query("count >= 2")
    .sort_values("mean", ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top_genres, x="mean", y="genre", palette="Reds_r", ax=ax)
ax.set_xlim(7, 9.5)
ax.set_xlabel("Average IMDb Score", fontsize=12)
ax.set_ylabel("Genre", fontsize=12)
ax.set_title("Top 10 Genres by Average IMDb Score (min. 2 titles)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7. Insights & Conclusion <a id="7"></a>

Let's summarize what we learned from the data.

In [ ]:
print("=" * 55)
print("  📊 KEY FINDINGS FROM THE NETFLIX DATASET")
print("=" * 55)

n_shows = (df_clean["type"] == "TV Show").sum()
n_movies = (df_clean["type"] == "Movie").sum()
print(f"\n📌 Dataset: {len(df_clean)} titles  ({n_shows} TV Shows, {n_movies} Movies)")

best_title = df_clean.loc[df_clean["imdb_score"].idxmax(), "title"]
best_score = df_clean["imdb_score"].max()
print(f"\n🏆 Highest-rated title : {best_title!r} (IMDb {best_score})")

avg_show = df_clean.loc[df_clean["type"] == "TV Show", "imdb_score"].mean()
avg_movie = df_clean.loc[df_clean["type"] == "Movie", "imdb_score"].mean()
print(f"\n📺 Avg IMDb score — TV Shows : {avg_show:.2f}")
print(f"🎬 Avg IMDb score — Movies   : {avg_movie:.2f}")

top_country = df_clean["country"].value_counts().idxmax()
print(f"\n🌍 Country with most titles: {top_country}")

most_common_rating = df_clean["rating"].value_counts().idxmax()
print(f"\n🔞 Most common age rating: {most_common_rating}")

avg_duration = movies["duration_min"].mean()
print(f"\n⏱  Average movie duration: {avg_duration:.0f} minutes")

corr = df_clean["votes"].corr(df_clean["imdb_score"])
print(f"\n📈 Votes ↔ IMDb Score correlation: {corr:.3f}")

print("\n" + "=" * 55)

---
## 8. Your Turn — Practice Exercises 🚀 <a id="8"></a>

Now it's your turn! Try solving these challenges on your own.

---

### Exercise 1 — Easy
Find the **5 lowest-rated** titles in the dataset. Display their `title`, `type`, `genre`, and `imdb_score`.

```python
# Your code here
```

---

### Exercise 2 — Easy
How many titles were added to Netflix in **2022**? How many in **2023**?

```python
# Your code here
```

---

### Exercise 3 — Medium
Create a bar chart showing the **average IMDb score per country** for the top 10 countries.

```python
# Your code here
```

---

### Exercise 4 — Medium
Filter for only **South Korean** titles. Which one has the highest IMDb score?  
*(Hint: use `df_clean[df_clean["country"] == "South Korea"]`)*

```python
# Your code here
```

---

### Exercise 5 — Hard
Create a **heatmap** showing the average IMDb score for each combination of `type` (TV Show / Movie) and `rating` (age rating).  
*(Hint: use `pd.pivot_table` and `sns.heatmap`)*

```python
# Your code here
```

---

### 💡 Bonus — Explore Further
- Which director has directed the most titles in this dataset?
- What month of the year sees the most new Netflix additions?
- Is there a trend in average IMDb scores over the release years?

In [ ]:
# Space for your answers!


---

## 🎉 Congratulations!

You have completed your **first data science project**! Here is what you practised:

| Skill | Tools Used |
|-------|------------|
| Loading data | `pd.read_csv()` |
| Inspecting data | `.head()`, `.dtypes`, `.describe()` |
| Handling missing values | `.isnull()`, `.fillna()`, `.dropna()` |
| Type conversion | `pd.to_datetime()`, `.str.extract()` |
| Grouping & aggregating | `.groupby()`, `.value_counts()` |
| Correlation | `.corr()` |
| Charts | `matplotlib`, `seaborn` |

### Next Steps
1. **Extend this project** — add your own questions and visualizations.
2. **Try machine learning** — predict IMDb scores using `scikit-learn`.
3. **Find more datasets** — [Kaggle](https://www.kaggle.com/datasets) has thousands of free datasets.
4. **Share your work** — push this notebook to GitHub and share with the community!